In [1]:
import os

print("Train images:",
      len(os.listdir("/kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/train")))

print("Val images:",
      len(os.listdir("/kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val")))

Train images: 1526
Val images: 169


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.0 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [3]:
yaml_content = """
path: /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive

train: images/train
val: images/val

nc: 1

names:
  0: license_plate
"""

with open("/kaggle/working/license_plate.yaml", "w") as f:
    f.write(yaml_content)

print("Done!")

Done!


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/kaggle/working/license_plate.yaml",
    epochs=100,
    imgsz=896,
    batch=32,
    device = [0,1],
    optimizer="AdamW",
    lr0=5e-4,

    cos_lr=True,

    # Augmentation
    degrees=5,
    translate=0.1,
    scale=0.3,

    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    mosaic=1.0,
    mixup=0.1,

    close_mosaic=15,
    workers = 4,
    weight_decay=1e-3,
    patience=30
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/license_plate.yaml, degrees=5, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=tor

In [5]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 366.0±170.3 MB/s, size: 188.5 KB)
val: Scanning /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/labels/val... 169 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 169/169 736.0it/s 0.2s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 4.4it/s 2.5s0.2s
                   all        169        169      0.993      0.994      0.995      0.866
Speed: 1.5ms preprocess, 6.8ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50: 0.9945857988165681
mAP50-95: 0.8658774801974989


In [6]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

model.predict(
    source="/kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val",
    save=True,
    conf=0.25
)


image 1/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video4_3860.jpg: 704x896 1 license_plate, 45.7ms
image 2/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video4_50.jpg: 512x896 1 license_plate, 42.7ms
image 3/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video5_0.jpg: 896x704 2 license_plates, 39.4ms
image 4/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video5_10.jpg: 896x896 1 license_plate, 11.8ms
image 5/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video5_100.jpg: 576x896 1 license_plate, 39.2ms
image 6/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video5_110.jpg: 576x896 1 license_plate, 8.3ms
image 7/169 /kaggle/input/datasets/ronakgohil/license-plate-dataset/archive/images/val/video5_120.jpg: 544x896 1 license_plate, 41.2ms
image 8/169 /kaggle/input/datasets/ronakgohil/license-pla

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'license_plate'}
 obb: None
 orig_img: array([[[244, 253, 255],
         [241, 250, 254],
         [241, 250, 254],
         ...,
         [ 82,  92,  86],
         [ 73,  83,  77],
         [ 62,  72,  66]],
 
        [[243, 252, 255],
         [238, 247, 251],
         [235, 244, 248],
         ...,
         [107, 117, 111],
         [112, 122, 116],
         [104, 114, 108]],
 
        [[243, 252, 255],
         [235, 244, 247],
         [228, 237, 240],
         ...,
         [142, 154, 148],
         [155, 167, 161],
         [138, 150, 144]],
 
        ...,
 
        [[162, 169, 178],
         [162, 169, 178],
         [162, 169, 178],
         ...,
         [116, 124, 141],
         [119, 127, 144],
         [108, 116, 133]],
 
        [[161, 168, 177],
         [162, 169, 178],
         [162, 169, 178],
         ...,
         [1

In [7]:
import shutil

shutil.make_archive(
    "/kaggle/working/runs",
    "zip",
    "/kaggle/working/runs"
)

print("Done!")

Done!
